In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# パスマネージャーを使ったトランスパイル

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
回路をトランスパイルする推奨方法は、ステージ付きパスマネージャーを作成し、回路を入力としてその `run` メソッドを実行することです。このページでは、この方法で量子回路をトランスパイルする方法について説明します。
## (ステージ付き)パスマネージャーとは？
Qiskit SDKの文脈において、トランスパイルとは入力回路を量子デバイスでの実行に適した形式に変換するプロセスを指します。トランスパイルは通常、トランスパイラーパスと呼ばれる一連のステップで行われます。回路は各トランスパイラーパスによって順番に処理され、あるパスの出力が次のパスへの入力となります。例えば、あるパスが回路内の連続する1量子ビットゲートをすべてマージし、次のパスがそれらのゲートをターゲットデバイスの基底ゲートセットに合成するといった流れになります。Qiskitに含まれるトランスパイラーパスは [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) モジュールにあります。

パスマネージャーは、トランスパイラーパスのリストを格納し、回路に対してそれらを実行できるオブジェクトです。パスマネージャーを作成するには、トランスパイラーパスのリストで [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) を初期化します。回路に対してトランスパイルを実行するには、回路を入力として [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) メソッドを呼び出します。

ステージ付きパスマネージャーは、通常のパスマネージャーよりも高い抽象レベルを表す特殊なパスマネージャーです。通常のパスマネージャーが複数のトランスパイラーパスで構成されるのに対し、ステージ付きパスマネージャーは複数の*パスマネージャー*で構成されます。これはトランスパイルが通常、[トランスパイラーのステージ](transpiler-stages)で説明されているように、各ステージがパスマネージャーで表される離散的なステージで行われるため、有用な抽象化です。ステージ付きパスマネージャーは [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) クラスで表されます。このページの残りの部分では、(ステージ付き)パスマネージャーの作成とカスタマイズ方法について説明します。
## プリセットのステージ付きパスマネージャーを生成する
合理的なデフォルト設定でプリセットのステージ付きパスマネージャーを作成するには、[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 関数を使用します。

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

回路または回路のリストをパスマネージャーでトランスパイルするには、回路または回路のリストを `run` メソッドに渡します。Hadamardゲートとそれに続く2つの隣接するCXゲートからなる2量子ビット回路で試してみましょう。

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

`generate_preset_pass_manager` 関数の引数の説明については、[トランスパイルのデフォルト設定と構成オプション](defaults-and-configuration-options)を参照してください。`generate_preset_pass_manager` への引数は [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) 関数への引数と一致します。

<CodeAssistantAdmonition tagLine="Having trouble remembering pass manager details? Try asking Qiskit Code Assistant." />

プリセットのパスマネージャーがニーズを満たさない場合は、(ステージ付き)パスマネージャーやトランスパイルパスを作成することでトランスパイルをカスタマイズできます。このページの残りの部分では、パスマネージャーの作成方法について説明します。トランスパイルパスの作成方法については、[独自のトランスパイラーパスを書く](custom-transpiler-pass)を参照してください。

## 独自のパスマネージャーを作成する

[qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) モジュールには、パスマネージャーの作成に使用できる多くのトランスパイラーパスが含まれています。パスマネージャーを作成するには、パスのリストで `PassManager` を初期化します。例えば、以下のコードは隣接する2量子ビットゲートをマージし、それらを [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate)、[$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)、[$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate) の基底ゲートセットに合成するトランスパイラーパスを作成します。

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

このパスマネージャーの動作を示すために、Hadamardゲートとそれに続く2つの隣接するCXゲートからなる2量子ビット回路でテストします。

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

回路にパスマネージャーを実行するには、`run` メソッドを呼び出します。

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

ダイナミカルデカップリングと呼ばれるエラー抑制技術を実装するパスマネージャーの作成方法を示す、より高度な例については、[ダイナミカルデカップリング用パスマネージャーの作成](dynamical-decoupling-pass-manager)を参照してください。

## ステージ付きパスマネージャーを作成する

`StagedPassManager` は個々のステージで構成されるパスマネージャーで、各ステージは `PassManager` インスタンスによって定義されます。`StagedPassManager` を作成するには、希望するステージを指定します。例えば、以下のコードは `init` と `translation` の2つのステージを持つステージ付きパスマネージャーを作成します。`translation` ステージは先ほど作成したパスマネージャーによって定義されます。

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

ステージ付きパスマネージャーに設定できるステージ数に制限はありません。

ステージ付きパスマネージャーを作成する別の有用な方法として、プリセットのステージ付きパスマネージャーから始めて一部のステージを入れ替える方法があります。例えば、以下のコードは最適化レベル3のプリセットパスマネージャーを生成し、カスタムの `pre_layout` ステージを指定します。

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[ステージジェネレーター関数](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions)は、カスタムパスマネージャーの構築に役立つ場合があります。
これらの関数は多くのパスマネージャーで使用される共通の機能を提供するステージを生成します。
例えば、[`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) は、レイアウトパスで選択された初期 `Layout` を指定のターゲットデバイスに「埋め込む」ステージの生成に使用できます。

## 次のステップ
> **Tip:** - [カスタムトランスパイラーパスを書く](custom-transpiler-pass)。
>     - [ダイナミカルデカップリング用パスマネージャーを作成する](dynamical-decoupling-pass-manager)。
>     - `generate_preset_passmanager` 関数の詳細については、[トランスパイルのデフォルト設定と構成オプション](defaults-and-configuration-options)のトピックを参照してください。
>     - [トランスパイラー設定の比較](/guides/circuit-transpilation-settings)ガイドを試してみてください。
>     - [トランスパイラーAPIドキュメント](https://docs.quantum.ibm.com/api/qiskit/transpiler)を確認してください。